# NB05 — Statistical Inference: t-tests, p-values, Confidence Intervals

> **This is the heart of the StatQuest video: "Is this slope real, or could it be zero by chance?"**

---

## The main ideas are:

1. Our estimated b1 is based on a **sample** — if we got new data, b1 would be slightly different
2. The **Standard Error (SE)** measures how much b1 varies from sample to sample
3. The **t-statistic** = b1 / SE asks: "how many standard errors away from zero is our estimate?"
4. The **p-value** answers: "if the true slope were zero, how often would we get a t-stat this large by chance?"
5. A **confidence interval** gives a range of plausible values for b1


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    """Draw a horizontal flow diagram.  steps = list of strings."""
    n = len(steps)
    default_colors = [
        '#1565C0','#2E7D32','#E65100','#6A1B9A',
        '#00695C','#AD1457','#37474F','#4E342E',
        '#0277BD','#558B2F',
    ]
    colors = (colors or default_colors[:n]) + default_colors
    notes  = notes or [''] * n

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n * 3.1)
    ax.set_ylim(-1.2, 2.4)
    ax.axis('off')

    bw, bh = 2.6, 1.3
    for i, (step, color, note) in enumerate(zip(steps, colors, notes)):
        x = i * 3.1
        box = FancyBboxPatch((x, 0.2), bw, bh,
                             boxstyle="round,pad=0.12",
                             facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.90)
        ax.add_patch(box)
        ax.text(x + bw/2, 0.2 + bh/2, step,
                ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold', multialignment='center')
        if note:
            ax.text(x + bw/2, 0.02, note,
                    ha='center', va='top', fontsize=7, color='#555', style='italic')
        if i < n - 1:
            ax.annotate('',
                xy=(x + bw + 0.38, 0.2 + bh/2),
                xytext=(x + bw + 0.08, 0.2 + bh/2),
                arrowprops=dict(arrowstyle='->', color='#444', lw=2.2))

    ax.set_title(title, fontsize=11, fontweight='bold', pad=6, color='#222')
    plt.tight_layout(pad=0.4)
    plt.show()

flow_diagram(
    steps=[
        'Null hypothesis:\nb1 = 0\n(no relationship)',
        'Estimate b1\nfrom data\n(OLS)',
        'Compute SE(b1):\nhow uncertain\nis b1?',
        't-stat =\nb1 / SE(b1)\nsignal/noise',
        'p-value:\nP(|t| > t_obs)\nif H0 true',
        'Decision:\np < 0.05?\n-> reject H0',
        '95% CI:\nb1 +/- t* * SE\nplausible range',
    ],
    title='NB05 Conceptual Flow: Statistical Inference for Regression Coefficients',
    colors=['#C62828','#1565C0','#2E7D32','#E65100','#6A1B9A','#00695C','#AD1457'],
    figsize=(16, 2.8),
)


## Step 1 — The Null Hypothesis

The null hypothesis is always:
```
H0: b1 = 0   (x has NO linear relationship with y)
```

We don't believe the null — we test it. We ask: *"If the slope really were zero, how surprising would our data be?"*

If the answer is "very surprising (p < 0.05)" -> we reject H0 -> the slope is statistically significant.


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(7)
n = 40
X = np.linspace(1, 10, n)
true_b1 = 2.5
y = true_b1*X + 5 + np.random.normal(0, 4, n)

xbar, ybar = X.mean(), y.mean()
b1 = np.sum((X-xbar)*(y-ybar)) / np.sum((X-xbar)**2)
b0 = ybar - b1*xbar

print(f"True slope: {true_b1}")
print(f"Estimated slope (b1): {b1:.4f}")
print(f"Null hypothesis: H0: b1 = 0")


## Step 2 — Standard Error of b1

**Key intuition (StatQuest style):**
- If you ran this experiment 1000 times and got a b1 each time, those b1s would form a distribution
- The **standard deviation of that sampling distribution** is the Standard Error
- SE measures uncertainty: large SE = b1 could be very different with new data

```
sigma^2 = SSR / (n-2)        <- estimated variance of residuals
SE(b1)  = sqrt(sigma^2 / sum(x - x_bar)^2)
```

Note: divides by **(n-2)** not n, because we estimated 2 parameters (b0 and b1).


In [ ]:
import numpy as np, scipy.stats as stats, matplotlib.pyplot as plt

y_hat  = b0 + b1*X
resid  = y - y_hat
n      = len(y)
df     = n - 2                                # degrees of freedom

sigma2 = np.sum(resid**2) / df               # unbiased residual variance
sxx    = np.sum((X-xbar)**2)

se_b1 = np.sqrt(sigma2 / sxx)
se_b0 = np.sqrt(sigma2 * (1/n + xbar**2/sxx))

print(f"Residual variance sigma^2 = {sigma2:.4f}")
print(f"SE(b0) = {se_b0:.4f}   (uncertainty in intercept)")
print(f"SE(b1) = {se_b1:.4f}   (uncertainty in slope)")
print()
print("Intuition: if SE(b1) is large, b1 could vary a lot with new data.")
print(f"Our b1 = {b1:.4f} is about {b1/se_b1:.1f} SE's away from zero.")


## Step 3 — The t-statistic

```
t = b1 / SE(b1)
```

Think of it as a **signal-to-noise ratio**:
- b1 is the signal (size of the effect)
- SE(b1) is the noise (uncertainty)
- t = signal / noise

Rule of thumb: |t| > 2 is roughly significant at the 5% level (for large n).


In [ ]:
# Compute t-statistics and visualise where they fall on the t-distribution
import numpy as np, scipy.stats as stats, matplotlib.pyplot as plt

t_b0 = b0 / se_b0
t_b1 = b1 / se_b1
p_b0 = 2 * stats.t.sf(abs(t_b0), df)    # two-sided p-value
p_b1 = 2 * stats.t.sf(abs(t_b1), df)

print(f"{'':15} {'Coef':>10}  {'SE':>10}  {'t-stat':>10}  {'p-value':>12}")
print("-"*62)
print(f"{'b0 (intercept)':15} {b0:>10.4f}  {se_b0:>10.4f}  {t_b0:>10.4f}  {p_b0:>12.6f}")
print(f"{'b1 (slope)':15}     {b1:>10.4f}  {se_b1:>10.4f}  {t_b1:>10.4f}  {p_b1:>12.6f}")

# Visualise the t-distribution and p-value for b1
t_vals = np.linspace(-6, 6, 400)
t_pdf  = stats.t.pdf(t_vals, df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_vals, t_pdf, 'b-', linewidth=2, label=f't-distribution (df={df})')
# Shade the p-value area
ax.fill_between(t_vals, t_pdf,
                where=(t_vals >= abs(t_b1)), color='crimson', alpha=0.4,
                label=f'p/2 (right tail)')
ax.fill_between(t_vals, t_pdf,
                where=(t_vals <= -abs(t_b1)), color='crimson', alpha=0.4,
                label=f'p/2 (left tail)')
ax.axvline(t_b1, color='crimson', linestyle='--', linewidth=2,
           label=f'Observed t = {t_b1:.2f}')
ax.axvline(-t_b1, color='crimson', linestyle='--', linewidth=2)
ax.set_xlabel('t value'); ax.set_ylabel('Probability density')
ax.set_title(f'p-value = {p_b1:.6f} = shaded area\n'
             f'"If H0: b1=0 were true, we would see |t| >= {abs(t_b1):.2f} only {p_b1*100:.4f}% of the time"')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Step 4 — Confidence Intervals

```
95% CI for b1:  [ b1 - t* * SE(b1),   b1 + t* * SE(b1) ]
```

where t* is the critical value (about 1.96 for large n).

**StatQuest interpretation:**
> "If we repeated this experiment 100 times, about 95 of the resulting CIs would contain the true slope."

**NOT:** "There is a 95% probability the true slope is in this interval."
(The true slope is a fixed number — it's either in the interval or it isn't.)

**Connection:** if the 95% CI does NOT include 0 -> p < 0.05 (they are mathematically equivalent).


In [ ]:
import numpy as np, scipy.stats as stats, matplotlib.pyplot as plt

t_crit = stats.t.ppf(0.975, df)    # t* for 95% CI, two-sided
print(f"t* (critical value, df={df}): {t_crit:.4f}")

ci_b0 = (b0 - t_crit*se_b0, b0 + t_crit*se_b0)
ci_b1 = (b1 - t_crit*se_b1, b1 + t_crit*se_b1)
print(f"95% CI for b0: [{ci_b0[0]:.3f}, {ci_b0[1]:.3f}]")
print(f"95% CI for b1: [{ci_b1[0]:.3f}, {ci_b1[1]:.3f}]")
print(f"True slope ({true_b1}) inside CI: {ci_b1[0] < true_b1 < ci_b1[1]}")
print(f"CI excludes 0: {ci_b1[0] > 0 or ci_b1[1] < 0}  -> same as p < 0.05")

# Simulate many experiments to verify "95 out of 100 CIs contain true value"
np.random.seed(42)
contains = 0
ci_list  = []
for _ in range(200):
    y_sim  = true_b1*X + 5 + np.random.normal(0, 4, n)
    ybar_s = y_sim.mean()
    b1_s   = np.sum((X-xbar)*(y_sim-ybar_s)) / sxx
    b0_s   = ybar_s - b1_s*xbar
    resid_s= y_sim - (b0_s + b1_s*X)
    sig2_s = np.sum(resid_s**2)/(n-2)
    se_s   = np.sqrt(sig2_s/sxx)
    lo, hi = b1_s - t_crit*se_s, b1_s + t_crit*se_s
    ci_list.append((lo, hi, lo < true_b1 < hi))
    if lo < true_b1 < hi: contains += 1

print(f"\nSimulation: {contains}/200 CIs contain the true slope ({contains/2:.1f}%)")
print("Expected: ~95%")

fig, ax = plt.subplots(figsize=(10, 5))
for i, (lo, hi, hit) in enumerate(ci_list[:80]):
    color = 'steelblue' if hit else 'crimson'
    ax.hlines(i, lo, hi, color=color, linewidth=1.5, alpha=0.7)
ax.axvline(true_b1, color='gold', linewidth=2.5, label=f'True slope = {true_b1}')
ax.set_xlabel('b1 value'); ax.set_ylabel('Simulation run')
ax.set_title(f'80 simulated 95% CIs — red ones MISS the true value (~5%)')
ax.legend(); ax.grid(alpha=0.3, axis='x'); plt.tight_layout(); plt.show()


In [ ]:
# Full statsmodels output — shows everything at once
import statsmodels.api as sm
import numpy as np

Xsm = sm.add_constant(X)
res = sm.OLS(y, Xsm).fit()
print(res.summary())
print("\nAnnotations:")
print("  coef      = estimated coefficient")
print("  std err   = SE of the coefficient")
print("  t         = coef / std_err")
print("  P>|t|     = two-sided p-value")
print("  [0.025 0.975] = lower and upper 95% CI bounds")


## Key Takeaways

| Term | Plain-English meaning |
|------|----------------------|
| SE(b1) | How much b1 would vary if we collected new data |
| t-stat | Signal-to-noise: how many SEs away from zero |
| p-value | "If H0 is true, how often do we see t this extreme?" |
| p < 0.05 | Statistically significant at 5% level |
| 95% CI | 95% of such intervals would contain the true b1 |
| CI excludes 0 | Exactly equivalent to p < 0.05 |

**Warning (StatQuest always says this):**
- p < 0.05 does NOT mean the effect is large
- p < 0.05 does NOT mean the result will replicate
- A large dataset can make tiny, useless effects "significant"

**Next: NB06 — the four assumptions OLS requires to be valid.**
